# SPY · Blind EMA rejection review
Run the notebook, then judge the rightmost completed candle. **Confirm rejection** locks the first close you would act on. **Wait one candle** reveals the next trading session. **No setup** closes the episode. Decide at the trigger or during the next five sessions.

Each chart shows 42 daily candles, EMA8, EMA21 and volume. Dates and dollar prices are hidden. The 25 episodes start at the first tagged close after more than five sessions since the previous tag. Their order is shuffled. The chart shows price indexed to the first visible close and volume relative to the largest visible bar.

After all decisions are locked, the report compares each confirmed close to the close five sessions later. Only 2020–2024 episodes and outcomes are scored. The 2019 bars provide indicator warm-up. A forecast ending after December 2024 is marked unavailable.

In [ ]:
# @title Load historical SPY episodes
from pathlib import Path
import random
import pandas as pd

SOURCE_COMMIT = "0d0a3cfe5acb9410382b5e58a855270620f46078"
DATA = f"https://raw.githubusercontent.com/jamelof23/SPY-EMA-V2/{SOURCE_COMMIT}/data"
price_columns = ["Open", "High", "Low", "Close", "Volume"]
def load_episodes(data):
    bars = pd.read_csv(f"{data}/source/SPY_daily_2019_2025.csv",
                       usecols=["Date", *price_columns]).set_index("Date").sort_index()
    bars = bars.loc["2019-01-01":"2024-12-31"].copy()
    assert bars.index.is_unique and bars.index[-1] == "2024-12-31"
    for span in (8, 21):
        bars[f"EMA{span}"] = bars.Close.ewm(span=span, adjust=False).mean()
    flags = pd.read_csv(f"{data}/labels/historical_audit_through_2025.csv",
                        usecols=["asof_date", "split", "bear_pullback_asof"])
    tagged = flags.loc[(flags.split == "train") & (flags.bear_pullback_asof == 1), "asof_date"]
    positions = sorted(bars.index.get_loc(day) for day in tagged)
    starts = [pos for i, pos in enumerate(positions) if i == 0 or pos - positions[i-1] > 5]
    assert len(starts) == 25 and all(41 <= p < len(bars) - 5 for p in starts)
    random.Random(42).shuffle(starts)
    return bars, starts

bars, starts = load_episodes(DATA)


In [ ]:
# @title Prepare chart and scoring functions
import io
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MaxNLocator

def draw_chart(end):
    # All plotted values, ranges and volume scaling use the visible window only.
    frame = bars.iloc[end-41:end+1]
    prices = frame.drop(columns="Volume") * (100 / frame.Close.iloc[0])
    volume = frame.Volume / frame.Volume.max()
    colors = np.where(prices.Close >= prices.Open, "#18806b", "#b74945")
    x = np.arange(len(frame))
    fig, (ax, vol) = plt.subplots(2, 1, sharex=True, figsize=(9.5, 4.05),
        gridspec_kw={"height_ratios": [4.4, 1], "hspace": .07})
    fig.subplots_adjust(left=.074, right=.985, bottom=.14, top=.90)
    for axis in (ax, vol):
        axis.grid(axis="y", color="#e4e8eb", linewidth=.65)
        axis.set_axisbelow(True)
        axis.tick_params(colors="#475569", length=0, labelsize=8)
        for spine in axis.spines.values():
            spine.set_visible(False)
    ax.vlines(x, prices.Low, prices.High, color=colors, linewidth=1.2, zorder=2)
    for i, (o, c) in enumerate(zip(prices.Open, prices.Close)):
        if o == c:
            ax.hlines(c, i-.33, i+.33, color=colors[i], linewidth=1.8)
        else:
            ax.add_patch(Rectangle((i-.33, min(o, c)), .66, abs(c-o),
                                   facecolor=colors[i], edgecolor=colors[i], linewidth=.5, zorder=3))
    for name, color in (("EMA8", "#1c67b3"), ("EMA21", "#cc801e")):
        ax.plot(x, prices[name], color=color, linewidth=1.65, label=name)
    ax.margins(y=.09)
    ax.legend(loc="lower right", bbox_to_anchor=(1,1), frameon=False, ncol=2, fontsize=9)
    ax.set_title("SPY  /  daily candles", loc="left", fontsize=12, fontweight="bold", color="#182c3e")
    ax.set_ylabel("Indexed price", fontsize=9)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    vol.bar(x, volume, color=colors, width=.66, alpha=.7)
    vol.set_ylabel("Volume", fontsize=9)
    vol.set_yticks([])
    vol.set_xlim(-1, 42)
    vol.set_xticks([0, 10, 20, 30, 41], ["−41", "−31", "−21", "−11", "Latest close"])
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=115, facecolor="white")
    plt.close(fig)
    return buffer.getvalue()

PREFIX = "SPY25v1-"
def replay(events):
    decisions, offset = [], 0
    for event in events:
        if len(decisions) >= len(starts) or event not in ("R","W","N"):
            raise ValueError("Invalid decision sequence.")
        if event == "W":
            if offset >= 5:
                raise ValueError("Already at the final candle in this episode.")
            offset += 1
        else:
            decisions.append((event, offset))
            offset = 0
    return decisions, offset

def score(decisions):
    assert len(decisions) == len(starts)
    rows = []
    for episode, (start, (action, offset)) in enumerate(zip(starts, decisions), 1):
        at = start + offset
        later = at + 5
        row = {"Episode":episode, "Trigger date":bars.index[start], "Decision date":bars.index[at],
               "Decision":"Confirmed" if action == "R" else "No setup",
               "Five-session end":"", "Direction":"", "Return (%)":np.nan}
        if action == "R":
            if later < len(bars):
                ret = 100*(bars.Close.iloc[later] / bars.Close.iloc[at] - 1)
                row.update({"Five-session end":bars.index[later], "Return (%)":ret,
                            "Direction":"DOWN" if ret < 0 else "UP" if ret > 0 else "FLAT"})
            else:
                row["Direction"] = "Outside review period"
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
# @title Open the blind review
import html
import base64
import ipywidgets as widgets
from IPython.display import display

state_path = Path.cwd() / "spy_ema_blind_replay_v1.txt"
events = []
def read_code(code):
    code = code.strip().upper()
    if not code.startswith(PREFIX):
        raise ValueError(f"The code should start with {PREFIX}")
    parsed = list(code[len(PREFIX):])
    replay(parsed)
    return parsed

if state_path.exists():
    events = read_code(state_path.read_text())

heading = widgets.HTML()
chart = widgets.Image(format="png", layout=widgets.Layout(width="min(100%, 920px)"))
confirm = widgets.Button(description="Confirm rejection", button_style="primary",
                         layout=widgets.Layout(width="150px"))
wait = widgets.Button(description="Wait one candle", layout=widgets.Layout(width="145px"))
skip = widgets.Button(description="No setup", layout=widgets.Layout(width="110px"))
controls = widgets.HBox([confirm, wait, skip], layout=widgets.Layout(flex_flow="row wrap"))
hint = widgets.HTML()
token = widgets.Text(description="Resume code", disabled=False, layout=widgets.Layout(width="78%"))
restore = widgets.Button(description="Restore progress")
message = widgets.HTML()
report = widgets.HTML()
def redraw():
    decisions, offset = replay(events)
    n = len(decisions)
    token.value = PREFIX + "".join(events)
    restore.disabled = bool(events)
    complete = n == len(starts)
    confirm.disabled = skip.disabled = complete
    wait.disabled = complete or offset == 5
    if complete:
        chart.layout.display = controls.layout.display = "none"
        heading.value = "<h3>Review complete · 25 episodes recorded</h3>"
        hint.value = ""
        result = score(decisions)
        counted = result.Direction.isin(["UP", "DOWN", "FLAT"])
        downs = int(result.Direction.eq("DOWN").sum())
        denominator = int(counted.sum())
        missing = int(result.Direction.eq("Outside review period").sum())
        absent = int(result.Decision.eq("No setup").sum())
        rate = f" ({downs/denominator:.0%})" if denominator else ""
        csv = base64.b64encode(result.to_csv(index=False).encode()).decode()
        report.value = (f"<p>SPY closed lower five sessions later in <b>{downs} of {denominator}</b>"
                        f"{rate} scorable confirmed rejections. No setup in {absent} episodes. "
                        f"Follow-up extended past 2024 for {missing} confirmed signals.</p>"
                        f'<a download="spy_ema_blind_results.csv" href="data:text/csv;base64,{csv}">Download results CSV</a>'
                        + result.to_html(index=False, na_rep="—", float_format=lambda v:f"{v:+.2f}"))
        return
    heading.value = (f'<h3 style="margin:0">Episode {n+1} of {len(starts)}'
                     f' &nbsp;·&nbsp; {"Trigger close" if offset == 0 else f"Close +{offset}"}</h3>')
    hint.value = ("<p>Choose on the rightmost completed candle. "
                  + ("This is the last close for this episode." if offset == 5
                     else "Wait reveals one more trading session.") + "</p>")
    chart.value = draw_chart(starts[n] + offset)

def save_events(new_events):
    replay(new_events)
    tmp = state_path.with_suffix(".tmp")
    tmp.write_text(PREFIX + "".join(new_events))
    tmp.replace(state_path)
    events[:] = new_events
    redraw()

def choose(action):
    try:
        message.value = ""
        save_events(events + [action])
    except (ValueError, OSError) as exc:
        message.value = html.escape(str(exc))

def restore_clicked(_):
    try:
        parsed = read_code(token.value)
        if events:
            raise ValueError("Restore is available only in a fresh review.")
        save_events(parsed)
        message.value = "Progress restored."
    except (ValueError, OSError) as exc:
        message.value = html.escape(str(exc))

for button, action in ((confirm,"R"), (wait,"W"), (skip,"N")):
    button.on_click(lambda _, choice=action: choose(choice))
restore.on_click(restore_clicked)
redraw()
display(widgets.VBox([heading, chart, hint, controls,
    widgets.HTML("<small>Keep the resume code if Colab times out. Paste it into a fresh review to continue.</small>"),
    widgets.HBox([token, restore]), message, report]))
